## Построение пайплайнов для загрузки параметров

### Загрузка всех датасетов

В начале загрузим все обработанные версии датасетов.

In [1]:
!pip install scikit-optimize
!pip install catboost
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 7.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, mean_squared_error, make_scorer
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')


In [4]:
df_adult = pd.read_csv('/content/adult_cleaned_new.csv')
df_bank = pd.read_csv('/content/bank_marketing_cleaned_new.csv')
df_cal = pd.read_csv('/content/california_housing_cleaned_new.csv')
df_sc = pd.read_csv('/content/superconductivity_cleaned_new.csv')
df_sp = pd.read_csv('/content/spambase_cleaned_new.csv')

In [5]:
datasets = {
    'Adult': df_adult,
    'Bank': df_bank,
    'California': df_cal,
    'Superconductivity': df_sc,
    'Spam': df_sp
}

In [7]:
for name, data in datasets.items():
  print(f'{name}:', end = '\n')
  print(len(data))

Adult:
48842
Bank:
45211
California:
20640
Superconductivity:
21263
Spam:
4601


In [8]:
for name, data in datasets.items():
  print(f'{name}:', end = '\n')
  print(data['target'].nunique())

Adult:
2
Bank:
2
California:
3842
Superconductivity:
3007
Spam:
2


### Создаем конфиги

**Конфиги датасетов**

В конфиге задаем:

1. Какой тип задачи решается в датасете
2. Колонку с таргетом
3. Колонки с категориальными и числовыми фичами
4. Флаг того, большой датасет или маленький (в зависимости от этого определяется число фолдов при оценке качества моделей).

In [9]:
DATASET_CONFIGS = {
    'Adult': {
        'dataframe': df_adult,
        'task': 'classification',
        'target_col': 'target',
        'categorical_cols': ['workclass', 'education', 'marital-status', 'occupation',
                             'relationship', 'race', 'sex', 'native-country'],
        'numeric_cols': ['age', 'fnlwgt', 'education-num', 'capital-gain',
                        'capital-loss', 'hours-per-week'],
        'size': 'large'
    },
    'Bank': {
        'dataframe': df_bank,
        'task': 'classification',
        'target_col': 'target',
        'categorical_cols': ['job', 'marital', 'education', 'contact',
                            'day_of_week', 'month', 'poutcome'],
        'numeric_cols': ['age', 'balance', 'campaign', 'pdays', 'previous',
                        'default', 'housing', 'loan', 'was_contacted'],
        'size': 'large'
    },
    'California': {
        'dataframe': df_cal,
        'task': 'regression',
        'target_col': 'target',
        'categorical_cols': ['ocean_proximity'],
        'numeric_cols': ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
                        'total_bedrooms', 'population', 'households', 'median_income',
                        'rooms_per_household', 'bedrooms_per_room', 'population_per_household'],
        'size': 'large'
    },
    'Superconductivity': {
        'dataframe': df_sc,
        'task': 'regression',
        'target_col': 'target',
        'categorical_cols': [],
        'numeric_cols': None,  # Все колонки кроме target
        'size': 'large'
    },
    'Spam': {
        'dataframe': df_sp,
        'task': 'classification',
        'target_col': 'target',
        'categorical_cols': [],
        'numeric_cols': None,  # Все колонки кроме target
        'size': 'small'
    }
}

**Конфиги моделей**

В конфиге задаем:

1. Какой тип модели используем
2. Словарь с перебираемыми параметрами
3. Список поддерживаемых задач

In [10]:
#Конфиги моделей с перебираемыми параметрами
MODEL_CONFIGS = {
    'LogisticRegression': {
        'class': LogisticRegression,
        'params': {
            'C': Real(0.01, 100, prior='log-uniform'),
            'penalty': Categorical(['l1', 'l2']),
            #'penalty': Categorical(['l1', 'l2']),
            #'solver': Categorical(['liblinear', 'saga']),
            'solver': Categorical(['liblinear']),
            'max_iter': Integer(100, 1000),
            'tol': [1e-3]
        },
        'supports': ['classification']
    },
    'Ridge': {
        'class': Ridge,
        'params': {
            'alpha': Real(0.001, 100, prior='log-uniform'),
            'solver': Categorical(['auto', 'svd', 'cholesky', 'lsqr', 'sag'])
        },
        'supports': ['regression']
    },
    'Lasso': {
        'class': Lasso,
        'params': {
            'alpha': Real(0.0001, 10, prior='log-uniform'),
            'max_iter': Integer(1000, 5000),
            'selection': Categorical(['cyclic', 'random'])
        },
        'supports': ['regression']
    },

    'ElasticNet': {
        'class': ElasticNet,
        'params': {
            'alpha': Real(0.001, 10, prior='log-uniform'),
            'l1_ratio': Real(0.1, 0.9),  # 0 = Ridge, 1 = Lasso
            'max_iter': Integer(1000, 5000)
        },
        'supports': ['regression']
    },
    'RandomForestClassifier': {
        'class': RandomForestClassifier,
        'params': {
            'n_estimators': Integer(50, 300),
            'max_depth': Integer(3, 20),
            'min_samples_split': Integer(2, 20),
            'min_samples_leaf': Integer(1, 10),
            'max_features': Categorical(['sqrt', 'log2', None])
        },
        'supports': ['classification']
    },
    'RandomForestRegressor': {
        'class': RandomForestRegressor,
        'params': {
            #'n_estimators': Integer(50, 500),
            'n_estimators': Integer(50, 300),
            'max_depth': Integer(3, 20),
            'min_samples_split': Integer(2, 20),
            'min_samples_leaf': Integer(1, 10),
            'max_features': Categorical(['sqrt', 'log2', None])
        },
        'supports': ['regression']
    },
    'LightGBMClassifier': {
        'class': lgb.LGBMClassifier,
        'params': {
            'n_estimators': Integer(50, 500),
            'max_depth': Integer(3, 15),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'num_leaves': Integer(10, 100),
            'min_child_samples': Integer(5, 50),
            'subsample': Real(0.6, 1.0),
            'colsample_bytree': Real(0.6, 1.0),
            'verbose': [-1]
        },
        'supports': ['classification']
    },
    'LightGBMRegressor': {
        'class': lgb.LGBMRegressor,
        'params': {
            'n_estimators': Integer(50, 500),
            'max_depth': Integer(3, 15),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'num_leaves': Integer(10, 100),
            'min_child_samples': Integer(5, 50),
            'subsample': Real(0.6, 1.0),
            'colsample_bytree': Real(0.6, 1.0),
            'verbose': [-1]
        },
        'supports': ['regression']
    },
    'CatBoostClassifier': {
        'class': CatBoostClassifier,
        'params': {
            'iterations': Integer(50, 500),
            'depth': Integer(3, 10),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'l2_leaf_reg': Real(1, 10),
            'border_count': Integer(32, 255),
            'verbose': [0]
        },
        'supports': ['classification']
    },
    'CatBoostRegressor': {
        'class': CatBoostRegressor,
        'params': {
            'iterations': Integer(50, 500),
            'depth': Integer(3, 10),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'l2_leaf_reg': Real(1, 10),
            'border_count': Integer(32, 255),
            'verbose': [0]
        },
        'supports': ['regression']
    }
}

### Пишем препроцессинг

In [11]:
def get_preprocessor(dataset_name, model_type):
    """
    Создает препроцессор в зависимости от датасета и типа модели.

    Args:
        dataset_name: Название датасета
        model_type: Тип модели ('linear', 'tree', 'boosting')
    Returns:
        ColumnTransformer или None
    """
    config = DATASET_CONFIGS[dataset_name]
    categorical_cols = config['categorical_cols']
    numeric_cols = config['numeric_cols']

    # Если все признаки числовые
    if not categorical_cols and numeric_cols is None:
        if model_type == 'linear':
            return RobustScaler()  # Устойчив к выбросам
        else:
            return None  # Деревьям скейлинг не нужен

    # Если есть категориальные признаки
    if model_type == 'linear':
        # Для линейных моделей: OneHotEncoder + RobustScaler
        return ColumnTransformer([
            ('num', RobustScaler(), numeric_cols if numeric_cols else
             [col for col in config['dataframe'].columns
              if col != config['target_col'] and col not in categorical_cols]),
            ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),
             categorical_cols)
        ])

    elif model_type == 'tree':
        return ColumnTransformer([
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
             categorical_cols)
        ], remainder='passthrough')

    elif model_type == 'boosting':
        # Для LightGBM/CatBoost: ничего не делаем, они сами обработают категории
        return None


def prepare_data(dataset_name):
    """
    Подготавливает X и y для датасета.
    """
    config = DATASET_CONFIGS[dataset_name]
    df = config['dataframe']
    target_col = config['target_col']

    X = df.drop(target_col, axis=1)
    y = df[target_col]

    # Для датасетов без явных числовых колонок берем все колонки
    if config['numeric_cols'] is None:
        config['numeric_cols'] = list(X.select_dtypes(include=[np.number]).columns)

    return X, y

Выберем дополнительно тип кросс-валидации в зависимости от датасета.

Также сделаем маппинг модели и типа задачи, которую она решает, с классом в sklearn.

In [12]:
def get_cv_splitter(dataset_name, task):
    """
    Выбирает тип кросс-валидации в зависимости от размера датасета и задачи.
    """
    config = DATASET_CONFIGS[dataset_name]

    if config['size'] == 'large':
        n_splits = 3
    else:
        n_splits = 3

    if task == 'classification':
        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    else:
        return KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [13]:
def get_model_for_task(model_name, task):
    """
    Подбирает правильную модель в зависимости от задачи.
    """
    mapping = {
        # Для классификации
        ('LogisticRegression', 'classification'): 'LogisticRegression',
        ('RandomForest', 'classification'): 'RandomForestClassifier',
        ('LightGBM', 'classification'): 'LightGBMClassifier',
        ('CatBoost', 'classification'): 'CatBoostClassifier',

        # Для регрессии
        #('LogisticRegression', 'regression'): 'Ridge',  # По умолчанию Ridge
        ('Ridge', 'regression'): 'Ridge',
        ('Lasso', 'regression'): 'Lasso',
        ('ElasticNet', 'regression'): 'ElasticNet',
        ('RandomForest', 'regression'): 'RandomForestRegressor',
        ('LightGBM', 'regression'): 'LightGBMRegressor',
        ('CatBoost', 'regression'): 'CatBoostRegressor'
    }

    return mapping.get((model_name, task))


def get_model_type(model_name):
    """
    Определяет тип модели для выбора препроцессора.
    """
    if model_name in ['LogisticRegression', 'Ridge', 'Lasso', 'ElasticNet']:
        return 'linear'
    elif model_name == 'RandomForest':
        return 'tree'
    elif model_name in ['LightGBM', 'CatBoost']:
        return 'boosting'
    else:
        # На всякий случай: если передано полное имя модели
        if 'Logistic' in model_name:
            return 'linear'
        elif 'Ridge' in model_name or 'Lasso' in model_name or 'ElasticNet' in model_name:
            return 'linear'
        elif 'RandomForest' in model_name:
            return 'tree'
        elif 'LightGBM' in model_name or 'CatBoost' in model_name:
            return 'boosting'
        else:
            raise ValueError(f"Неизвестный тип модели: {model_name}")

In [14]:
## Функция для логирования
import csv
import os

def log_best_to_csv(log_file, iteration, best_score, best_params, task, metric_name):
    """Дописывает строку в CSV-файл логов."""
    file_exists = os.path.isfile(log_file)
    with open(log_file, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['iteration', 'best_score', 'best_params', 'task', 'metric'])
        writer.writerow([iteration, best_score, best_params, task, metric_name])

### Основная функция для оптимизации

Напишем функцию, которая принимает на вход:

1. Датасет
2. Модель
3. Число Итераций

И ищет наилучшие параметры модели по байесовскому подходу, используя гауссовский случайный процесс.

In [15]:
def run_optuna_optimization(dataset_name, model_name, n_trials=50,
                            log_interval=5, log_file=None):
    """
    Запуск байесовской оптимизации гиперпараметров с помощью Optuna (TPE).
    Поддерживает LogisticRegression, RandomForest, LightGBM, Ridge, Lasso, ElasticNet.
    """
    dataset_config = DATASET_CONFIGS[dataset_name]
    task = dataset_config['task']

    full_model_name = get_model_for_task(model_name, task)
    if full_model_name not in MODEL_CONFIGS:
        raise ValueError(f"Модель {model_name} не поддерживает задачу {task}")

    model_config = MODEL_CONFIGS[full_model_name]
    model_class = model_config['class']

    print(f"\n{'='*70}")
    print(f"Optuna оптимизация: {model_name} на {dataset_name}")
    print(f"Задача: {task.upper()}")
    print(f"{'='*70}")

    X, y = prepare_data(dataset_name)
    print(f"Размер данных: X={X.shape}, y={y.shape}")

    if log_file:
        with open(log_file, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['iteration', 'best_score', 'best_params', 'task', 'metric'])

    # Для LightGBM нужно преобразовать категориальные колонки в 'category' dtype
    if model_name == 'LightGBM':
        categorical_cols = dataset_config['categorical_cols']
        for col in categorical_cols:
            if col in X.columns:
                X[col] = X[col].astype('category')
        print(f"LightGBM: категориальные колонки преобразованы в category dtype.")

    model_type = get_model_type(model_name)
    preprocessor = get_preprocessor(dataset_name, model_type)

    if preprocessor is not None:
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model_class())
        ])
    else:
        pipeline = Pipeline([
            ('model', model_class())
        ])

    # Метрика и кросс-валидация
    scoring = 'roc_auc' if task == 'classification' else 'neg_mean_squared_error'
    metric_name = 'ROC-AUC' if task == 'classification' else 'MSE'
    cv = get_cv_splitter(dataset_name, task)
    print(f"Кросс-валидация: {cv.get_n_splits()}-fold")
    print(f"Метрика: {metric_name}")

    # Параметры для Optuna
    param_space = model_config['params']

    def objective(trial):
        params = {}
        for pname, pspace in param_space.items():
            if isinstance(pspace, Real):
                params[pname] = trial.suggest_float(pname, pspace.low, pspace.high, log=(pspace.prior=='log-uniform'))
            elif isinstance(pspace, Integer):
                params[pname] = trial.suggest_int(pname, pspace.low, pspace.high)
            elif isinstance(pspace, Categorical):
                params[pname] = trial.suggest_categorical(pname, pspace.categories)
            # фиксированные параметры (например, verbose) игнорируются

        # Установка параметров в пайплайне
        pipeline.set_params(**{f'model__{key}': val for key, val in params.items()})

        # Кросс-валидация
        scores = cross_val_score(pipeline, X, y, cv=cv, scoring=scoring, n_jobs=1, error_score='raise')
        return np.mean(scores)

    # Создание study с TPE сэмплером
    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction='maximize', sampler=sampler)

    print(f"\nЗапуск оптимизации ({n_trials} trials)...")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_score = study.best_value
    best_params = study.best_params

    print(f"\n{'='*70}")
    print(f"РЕЗУЛЬТАТЫ:")
    print(f"Лучшее значение метрики: {best_score:.4f}")
    print(f"Лучшие параметры: {best_params}")
    print(f"{'='*70}")

    # Логирование в CSV (накопленный лучший результат каждые log_interval итераций)
    if log_file:
        trials_df = study.trials_dataframe()
        # Убедимся, что значения идут в порядке номеров trials
        best_so_far = -np.inf
        best_params_so_far = None
        written_iterations = set()
        for idx, row in trials_df.iterrows():
            trial_num = row['number'] + 1  # trials нумеруются с 0
            score = row['value']
            if score > best_so_far:
                best_so_far = score
                best_params_so_far = {k: v for k, v in study.trials[int(row['number'])].params.items()}
            if trial_num % log_interval == 0 and trial_num not in written_iterations:
                log_best_to_csv(log_file, trial_num, best_so_far, best_params_so_far, task, metric_name)
                written_iterations.add(trial_num)

    return {
        'dataset': dataset_name,
        'model': model_name,
        'task': task,
        'best_score': best_score,
        'best_params': best_params,
        'study': study
    }

Возможные модели:

1. LogisticRegression
2. Ridge
3. Lasso
4. ElasticNet
5. RandomForest
6. LightGBM
7. CatBoost

Возможные датасеты:

1. Adult
2. Bank
3. California
4. Superconductivity
5. Spam

Добавим логирование на Гугл Диск

In [16]:
from google.colab import drive
import os

# Монтируем Google Диск
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
# Задаём папку для логов на Диске (можно изменить название)
base_log_dir = '/content/drive/MyDrive/ЦУ Курсы/Метопты/Optuna_logs'
os.makedirs(base_log_dir, exist_ok=True)  # создаём, если не существует

Запустим функцию

In [ ]:
log_interval = 5
log_file = os.path.join(base_log_dir, 'optuna_RandomForest_Spam_logs.csv')

test_results = run_optuna_optimization('Spam', 'RandomForest', n_trials = 50,
                                        log_interval = log_interval, log_file = log_file)

[I 2026-06-01 11:51:21,740] A new study created in memory with name: no-name-db7ecd01-feb1-48f5-8c0c-9d7f65a0f01c



Optuna оптимизация: RandomForest на Spam
Задача: CLASSIFICATION
Размер данных: X=(4601, 57), y=(4601,)
Кросс-валидация: 3-fold
Метрика: ROC-AUC

Запуск оптимизации (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-01 11:51:24,959] Trial 0 finished with value: 0.9807244265828908 and parameters: {'n_estimators': 144, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9807244265828908.
[I 2026-06-01 11:51:31,688] Trial 1 finished with value: 0.9817549717953726 and parameters: {'n_estimators': 267, 'max_depth': 13, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.9817549717953726.
[I 2026-06-01 11:51:38,232] Trial 2 finished with value: 0.9636412279417067 and parameters: {'n_estimators': 95, 'max_depth': 6, 'min_samples_split': 7, 'min_samples_leaf': 6, 'max_features': None}. Best is trial 1 with value: 0.9817549717953726.
[I 2026-06-01 11:51:39,651] Trial 3 finished with value: 0.9775097338390811 and parameters: {'n_estimators': 85, 'max_depth': 8, 'min_samples_split': 8, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.9817549717953726.


### Поиск по одной модели для всех датасетов

In [18]:
def optimize_model_on_all_datasets_optuna(model_name, n_trials=50,
                                          log_interval=5, base_log_dir='logs'):
    """
    Запускает Optuna оптимизацию для выбранной модели на всех совместимых датасетах.
    """
    os.makedirs(base_log_dir, exist_ok=True)
    all_datasets = list(DATASET_CONFIGS.keys())

    for dataset_name in all_datasets:
        task = DATASET_CONFIGS[dataset_name]['task']
        if get_model_for_task(model_name, task) is None:
            print(f"Пропускаем {dataset_name} – модель {model_name} не поддерживает задачу {task}")
            continue

        log_filename = f"optuna_{model_name}_{dataset_name}_logs.csv"
        log_filepath = os.path.join(base_log_dir, log_filename)

        print(f"\n====== Запуск {model_name} на {dataset_name} ======")
        run_optuna_optimization(
            dataset_name=dataset_name,
            model_name=model_name,
            n_trials=n_trials,
            log_interval=log_interval,
            log_file=log_filepath
        )

In [19]:
optimize_model_on_all_datasets_optuna('Lasso', n_trials=50, log_interval=5, base_log_dir='/content/drive/MyDrive/ЦУ Курсы/Метопты/Optuna_logs')

[I 2026-06-02 18:09:37,950] A new study created in memory with name: no-name-fd47c0df-1534-43c2-99be-010cdde9e441


Пропускаем Adult – модель Lasso не поддерживает задачу classification
Пропускаем Bank – модель Lasso не поддерживает задачу classification

====== Запуск Lasso на California ======

Optuna оптимизация: Lasso на California
Задача: REGRESSION
Размер данных: X=(20640, 12), y=(20640,)
Кросс-валидация: 3-fold
Метрика: MSE

Запуск оптимизации (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-02 18:09:44,520] Trial 0 finished with value: -4769209821.049946 and parameters: {'alpha': 0.0074593432857265485, 'max_iter': 4803, 'selection': 'cyclic'}. Best is trial 0 with value: -4769209821.049946.
[I 2026-06-02 18:09:48,330] Trial 1 finished with value: -4769210247.401996 and parameters: {'alpha': 0.0006026889128682511, 'max_iter': 1624, 'selection': 'random'}. Best is trial 0 with value: -4769209821.049946.
[I 2026-06-02 18:09:49,050] Trial 2 finished with value: -4769204036.596655 and parameters: {'alpha': 0.10129197956845731, 'max_iter': 3832, 'selection': 'random'}. Best is trial 2 with value: -4769204036.596655.
[I 2026-06-02 18:09:49,843] Trial 3 finished with value: -4769130319.281255 and parameters: {'alpha': 1.452824663751602, 'max_iter': 1849, 'selection': 'random'}. Best is trial 3 with value: -4769130319.281255.
[I 2026-06-02 18:09:52,373] Trial 4 finished with value: -4769210078.047488 and parameters: {'alpha': 0.0033205591037519565, 'max_iter': 3099, 'se

[I 2026-06-02 18:10:28,577] A new study created in memory with name: no-name-bed2d92b-fb7b-46b9-87b6-b499335a4262


[I 2026-06-02 18:10:28,424] Trial 49 finished with value: -4769205917.601571 and parameters: {'alpha': 0.07071757800029692, 'max_iter': 2731, 'selection': 'cyclic'}. Best is trial 11 with value: -4769011544.538087.

РЕЗУЛЬТАТЫ:
Лучшее значение метрики: -4769011544.5381
Лучшие параметры: {'alpha': 6.1685918911184485, 'max_iter': 2491, 'selection': 'random'}

====== Запуск Lasso на Superconductivity ======

Optuna оптимизация: Lasso на Superconductivity
Задача: REGRESSION
Размер данных: X=(21263, 81), y=(21263,)
Кросс-валидация: 3-fold
Метрика: MSE

Запуск оптимизации (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-02 18:10:51,796] Trial 0 finished with value: -314.3800542556172 and parameters: {'alpha': 0.0074593432857265485, 'max_iter': 4803, 'selection': 'cyclic'}. Best is trial 0 with value: -314.3800542556172.
[I 2026-06-02 18:11:03,440] Trial 1 finished with value: -313.0992326612432 and parameters: {'alpha': 0.0006026889128682511, 'max_iter': 1624, 'selection': 'random'}. Best is trial 1 with value: -313.0992326612432.
[I 2026-06-02 18:11:06,772] Trial 2 finished with value: -346.13867403810127 and parameters: {'alpha': 0.10129197956845731, 'max_iter': 3832, 'selection': 'random'}. Best is trial 1 with value: -313.0992326612432.
[I 2026-06-02 18:11:07,716] Trial 3 finished with value: -425.6627192363065 and parameters: {'alpha': 1.452824663751602, 'max_iter': 1849, 'selection': 'random'}. Best is trial 1 with value: -313.0992326612432.
[I 2026-06-02 18:11:31,507] Trial 4 finished with value: -312.95071360169794 and parameters: {'alpha': 0.0033205591037519565, 'max_iter': 3099, '

### Архив